# Correr en GPU todo lo pendiente

Este cuaderno repite, **con más datos y más entrenamiento**, las mediciones que en CPU quedaron cortas, y prueba las piezas más prometedoras con datos reales de Hugging Face. Al final deja todos los informes en un `.zip` en tu Google Drive.

**Cómo usarlo**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4 alcanza).
2. *Entorno de ejecución → Ejecutar todo*. Cada sección dice cuánto tarda; puedes saltarte las que no quieras.
3. Al terminar, comparte el enlace del `.zip` (queda en `Mi unidad/Notbook/resultados_gpu.zip`) para integrar las cifras en el README y en las recomendaciones.

> El repo es privado. Guarda un token de GitHub de **solo lectura** en *Secretos de Colab* (ícono de llave) con el nombre `GITHUB_TOKEN`. **Nunca pegues llaves en una celda.**

## 1. Preparar (≈ 3 min)

In [ ]:
import os, subprocess, torch
from google.colab import drive, userdata

drive.mount("/content/drive")
SALIDA = "/content/drive/MyDrive/Notbook/resultados_gpu"
os.makedirs(SALIDA, exist_ok=True)
os.environ["HF_HOME"] = "/content/drive/MyDrive/Notbook/hf"          # las descargas quedan para la próxima vez
os.environ["LABORATORIO_CACHE"] = "/content/drive/MyDrive/Notbook/cache"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # opcional
except Exception:
    pass

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt "stable-baselines3>=2.9.0"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO hay GPU: cambia el tipo de entorno")

## 2. Video: tripletas contra U-Net con más datos (≈ 20-40 min)

La pista más fuerte del laboratorio. En CPU se midió con 2 000 clips y 20 épocas; aquí, 10 000 clips y 40 épocas.

In [ ]:
!python -m laboratorio.banco --video --entreno 10000 --prueba 1000 --epocas 40 --semillas 3 --salida $SALIDA/video

## 3. Datos en tabla y combinar modelos (≈ 10 min)

Neurona dendrítica, DMNA y genética invertida en la tarea caótica, y el stacking contra la votación ponderada y Gradient Boosting.

In [ ]:
!python -m laboratorio.banco --tarea caotica_20d --pieza dendritica --pieza dmna --pieza genetica_invertida --pieza dropout --semillas 5 --salida $SALIDA/tabla
!python -m laboratorio.banco --clasicos --salida $SALIDA/tabla

## 4. Imágenes y texto reales (≈ 1-2 h)

Las piezas prometedoras dentro de las arquitecturas de la industria (CNN, TextCNN, Transformer), con MNIST, CIFAR-10 y reseñas en español.

In [ ]:
!python -m laboratorio.banco --competencia --tipo imagen --pieza dendritica --pieza dmna --pieza genetica_invertida --pieza dropout --salida $SALIDA/imagen
!python -m laboratorio.banco --competencia --tipo texto --pieza dendritica --pieza dmna --pieza dropout --salida $SALIDA/texto

## 5. Texto del cuaderno 01 (≈ 1-2 h)

Corrector GNN con residuales, tokenizadores (sílabas y el de refuerzo contra BPE y GPT-4o) y la difusión de texto con más entrenamiento.

In [ ]:
!python -m laboratorio.banco --correccion --semillas 3 --salida $SALIDA/texto01
!python -m laboratorio.banco --tokenizadores --tarea resenas_es --semillas 3 --salida $SALIDA/texto01
!python -m laboratorio.banco --difusion --entreno 20000 --pasos 6000 --semillas 2 --salida $SALIDA/texto01

## 6. D-ReG y HCMG con más entrenamiento (≈ 30 min)

In [ ]:
!python -m laboratorio.banco --generativos07 --epocas 40 --pasos 3000 --semillas 3 --salida $SALIDA/generativos07

## 7. Entornos de refuerzo (opcional, ≈ 1 h; usa sobre todo la CPU)

In [ ]:
!python -m laboratorio.banco --refuerzo --pasos 300000 --semillas 3 --salida $SALIDA/refuerzo

## 8. Guardar todo en un `.zip`

In [ ]:
import shutil
ruta = shutil.make_archive("/content/drive/MyDrive/Notbook/resultados_gpu", "zip", SALIDA)
print("Listo:", ruta, "→ compártelo para integrar los resultados")